# CNC AI Assistant - Fine-Tuning Pipeline
**Kevin Danh** | DSC 670: Advanced Uses of Generative AI | Bellevue University
---

## Project Overview

The goal of this project is to fine-tune a generative AI model to better understand CNC programming and manufacturing data. Specifically, I want the model to explain G-code, identify common programming mistakes, and provide insights from machining-related datasets such as tool wear, cycle time, and error counts.

I chose this topic because CNC machining is widely used in manufacturing industries such as aerospace, and writing G-code can be difficult for people who are not experienced programmers or machinists. By fine-tuning a model on CNC-specific examples, I hope to create a more specialized assistant that can provide clearer and more consistent responses than a general-purpose language model.

In [3]:
# Import necessary libraries
import os
import json
import time
from openai import OpenAI

## Importing Required Libraries

This notebook uses four libraries: os and dotenv for secure API keymanagement, json for dataset validation, time for tracking training 
duration, and the OpenAI Python SDK for all API interactions including file upload, job creation, and model testing.

In [3]:
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("OPENAI_KEY")
print("API key loaded successfully.")

API key loaded successfully.


## Loading the OpenAI API Key

To keep my API key secure, I stored it in my environment rather than writing it directly in the notebook. The key is read into Python and used to authenticate all requests to the OpenAI API.

In [2]:
# Training data preview
with open("training_data.jsonl") as f:
    for i, line in enumerate(f):
        if i >= 3:
            break
        print(json.dumps(json.loads(line), indent=2))
        print("---")

{
  "messages": [
    {
      "role": "system",
      "content": "You are a CNC G-code assistant specializing in Fanuc-style milling programs. Explain code clearly, identify safety issues, and help interpret machining data."
    },
    {
      "role": "user",
      "content": "Explain this G-code block line by line:\nG21\nG90\nG17\nG54\nT01 M06\nM03 S1200\nG00 X0 Y0\nG00 Z5.0\nG01 Z-2.0 F100\nG01 X50.0 F200\nM05\nG00 Z50.0\nM30"
    },
    {
      "role": "assistant",
      "content": "Here is a line-by-line breakdown of this program:\n\nG21 \u2014 Sets the control to metric (millimeter) input mode. All coordinates will be interpreted in mm.\nG90 \u2014 Activates absolute positioning mode. All X, Y, Z values are measured from the work coordinate origin.\nG17 \u2014 Selects the XY plane for circular interpolation and cutter compensation. This is the standard plane for vertical milling.\nG54 \u2014 Activates work coordinate offset #1. The machine will reference all moves from the zero po

In [11]:
# Quick validation of Claude's generated JSONL dataset
with open("training_data.jsonl") as f:
    for i,line in enumerate(f):
        json.loads(line)

print("Valid JSONL")

Valid JSONL


## Validating the Training Dataset
Because no large public dataset for CNC G-code fine-tuning existed, I generated synthetic training examples using Claude (Anthropic) and manually reviewed and edited each one for technical accuracy and consistency with Fanuc conventions.

Before uploading the dataset, I verified that each line in the JSONL file could be successfully parsed as valid JSON. This step is important because fine-tuning requires the training file to follow a strict format. Catching formatting issues early helps prevent errors during file upload and model training.

In [7]:
# Create an instance of the OpenAI client
client = OpenAI(api_key=api_key)

## Creating the OpenAI Client

After loading the API key, I created an instance of the OpenAI client. This object provides access to the API methods used throughout the project, including file uploads, fine-tuning job creation, status monitoring, and model testing.

In [8]:
# Upload the training file
training_file = client.files.create(
    file=open("training_data.jsonl", "rb"),
    purpose="fine-tune"
)

training_file.id

'file-7R59gVt9YFF7ZXeorMWo2E'

## Uploading the Training File

The training dataset was uploaded to OpenAI using the Files API with the purpose set to `fine-tune`. This makes the file available for use when creating the fine-tuning job.

In [9]:
# Start timer
start_time = time.time()

job = client.fine_tuning.jobs.create(
    training_file=training_file.id,
    model="gpt-4o-mini-2024-07-18"
)

print("Fine-tuning job started.")
print("Job ID:", job.id)

Fine-tuning job started.
Job ID: ftjob-qHIvWYCzWQInKHXPz7pSzS9S


## Creating the Fine-Tuning Job

I selected GPT-4o Mini (2024-07-18) as the base model because it offers a strong balance between performance and cost. This model is well suited for structured technical tasks such as explaining G-code and identifying programming errors. The fine-tuning job uses the uploaded dataset to adapt the model to CNC and manufacturing-specific tasks.

In [10]:
# Poll until job finishes
previous_status = None

while True:
    status = client.fine_tuning.jobs.retrieve(job.id)

    if status.status != previous_status:
        print("Status:", status.status)
        print("Fine-tuned model:", status.fine_tuned_model)
        print("-" * 50)
        previous_status = status.status

    if status.status in ["succeeded", "failed", "cancelled"]:
        break

    time.sleep(180) # Check every 3 minutes

# End timer
end_time = time.time()

# Calculate elapsed time
elapsed_seconds = end_time - start_time
elapsed_minutes = elapsed_seconds / 60

print("\nTraining Complete")
print(f"Total elapsed time: {elapsed_seconds:.2f} seconds")
print(f"Total elapsed time: {elapsed_minutes:.2f} minutes")

Status: validating_files
Fine-tuned model: None
--------------------------------------------------
Status: running
Fine-tuned model: None
--------------------------------------------------
Status: succeeded
Fine-tuned model: ft:gpt-4o-mini-2024-07-18:personal::DeCanFUu
--------------------------------------------------

Training Complete
Total elapsed time: 1449.41 seconds
Total elapsed time: 24.16 minutes


## Monitoring the Fine-Tuning Process

After submitting the job, I periodically checked its status using the OpenAI API. To avoid generating excessive notebook output, the status was only printed when it changed, and the script waited three minutes between checks. This approach kept the notebook concise while still documenting the progression of the training job.

## What Fine-Tuning Does Under the Hood

Fine-tuning is the process of taking a pretrained large language model (LLM) and further training it on a smaller, task-specific dataset so that it becomes better at performing a particular type of task. In this project, the base model already has broad knowledge of programming, manufacturing, and natural language. However, it has not been specifically optimized to explain CNC G-code, identify machining-related errors, or interpret manufacturing datasets in the exact style and level of detail I want. Fine-tuning adjusts the model so that these types of responses become more consistent and domain focused.

At a high level, the model is made up of billions of numerical parameters, often called weights. These weights determine how the model predicts the next token in a sequence of text. During fine-tuning, OpenAI presents each example from the training dataset to the model. Each example contains a prompt (such as a block of G-code or a machining dataset) and an ideal response. The model first generates internal probability estimates for the next tokens in the expected answer. These predictions are compared to the correct tokens in the training example, and a loss value is calculated to measure how far the model's predictions are from the desired response.

Using an optimization algorithm based on gradient descent, the model computes how each parameter contributed to the prediction error. The algorithm then makes small adjustments to many of the weights in directions that reduce the loss. This process is repeated across all examples in the dataset, often for multiple passes called epochs. Over time, the model becomes more likely to generate responses that resemble the examples it was trained on. In other words, it "learns" the patterns, terminology, structure, and style contained in the dataset.

In this CNC project, fine-tuning teaches the model several specialized behaviors. It learns that certain G-code sequences are considered safer than others, that commands such as `M03` should typically appear before cutting moves, and that high tool wear is often associated with longer cycle times and increased error rates. It also learns how I want explanations to be written: technically accurate, clearly structured, and understandable to someone who may not be an expert machinist.

An important point is that fine-tuning does not replace the model's general knowledge. The model still retains its broad understanding of language, coding, and reasoning that it acquired during its original large-scale pretraining. Fine-tuning only nudges the existing weights so that the model becomes more specialized in the areas represented by the training examples. This is why relatively small datasets can have a noticeable effect when the examples are consistent and high quality.

The result of the process is a new model identifier, separate from the original base model. The base model remains unchanged, while the fine-tuned model contains adjusted parameters that reflect the patterns learned from the custom dataset. This new model can then be used like any other OpenAI model, but it will respond in a way that is more aligned with the CNC programming and manufacturing analysis tasks defined in this project.

In [12]:
# Save the fine-tuned model name
fine_tuned_model = status.fine_tuned_model
print("Fine-tuned Model:", fine_tuned_model)

Fine-tuned Model: ft:gpt-4o-mini-2024-07-18:personal::DeCanFUu


In [13]:
# Check on the tokens used for pricing info
print("Trained Tokens:", status.trained_tokens)
print("Result Files:", status.result_files)
print("Training Completed At:", status.finished_at)

Trained Tokens: 75837
Result Files: ['file-76g2xBaZexu7SpmpdNanHt']
Training Completed At: 1778473893


## Fine-Tuned Model

The completed fine-tuning job produced the following model used in the Streamlit application:

`ft:gpt-4o-mini-2024-07-18:personal::DeCanFUu`

- **Trained tokens:** 75,837
- **Training time:** 24.16 minutes
- **Training file ID:** `file-7R59gVt9YFF7ZXeorMWo2E`

## Training Metrics and Cost

I used Python's `time` module to record how long the complete fine-tuning process took. This included submitting the job, waiting for the training to finish, and retrieving the final model. Tracking the elapsed time provided useful information about the computational effort required to train the model.

After the fine-tuning job completed, I examined several metrics returned by the OpenAI API. The most important value for understanding cost was the number of trained tokens. In fine-tuning, tokens represent small pieces of text that the model processes during training. The total number of trained tokens depends on both the size of the dataset and the number of training epochs performed by the service.

I recorded the trained token count so that I could estimate the approximate cost of the fine-tuning job using OpenAI's pricing information. Because GPT-4o Mini has a relatively low training cost, even a dataset containing dozens of CNC and manufacturing examples can be fine-tuned for a modest amount. Tracking token usage helped me better understand how dataset size directly affects training expense and reinforced the importance of balancing model quality with cost efficiency.

In addition to trained tokens, I also recorded the result files generated by the training process and the timestamp indicating when the job finished. These outputs provide documentation of the completed fine-tuning process and can be useful for reviewing training metrics or reproducing the experiment later.

In [14]:
completion = client.chat.completions.create(
    model=fine_tuned_model,
    messages=[
        {
            "role": "system",
            "content": "You are a CNC G-code assistant that explains and corrects machining programs."
        },
        {
            "role": "user",
            "content": "Explain what is wrong with this G-code:\nG21\nT02 M06\nG01 X50 Y50 F200\nM03 S800\nM30"
        }
    ]
)

print(completion.choices[0].message.content)

This program has several critical issues:

1. No modal positioning block before the tool change (M06):
G01 (linear move) is a commanded G-code, it is not a modal state. When M06 is executed, the machine enters a tool change state where all modal geometry (like the last G01 move) is canceled. The control will not execute G01 X50 Y50 as expected after tool change; the machine will either move rapidly in a safe return way or remain stationary, depending on the machine's M06 behavior. This is a fundamental programming error — you cannot use a linear motion command as the first line after M06.

2. Tool change order is wrong:
The M06 command (tool change) must occur after the spindle is at speed and ready to cut and after a rapid positioning move to the tool change position if necessary. The current order has the machine trying to move to a work position before the spindle is started and before the tool is even in the spindle.

3. M03 (spindle on) is commanded after the positioning move:
The

## Testing the Fine-Tuned Model

After training completed, I tested the fine-tuned model using a prompt containing flawed G-code. The purpose of this test was to determine whether the model could identify unsafe programming practices and explain them clearly. This evaluation helps show whether fine-tuning improved the model's domain-specific understanding.

## Results and Evaluation

The fine-tuned model successfully generated explanations of CNC G-code and identified several programming issues, such as feed movements occurring before the spindle was started. Compared with the base model, the responses were more focused on CNC terminology and safety considerations.

Overall, the project demonstrated that fine-tuning can be used to adapt a general language model to a specialized manufacturing domain. Although the model is not intended to replace experienced machinists, it shows strong potential as an educational and decision-support tool for explaining G-code and interpreting machining data.

## Final Reflection

This milestone helped me better understand the complete fine-tuning workflow using OpenAI. I learned how to create and validate a custom dataset, upload it to the API, monitor the training process, and test the resulting model. I also gained a better appreciation for how generative AI can be applied to technical areas such as CNC machining and manufacturing analytics.

If I were to continue this project, I would expand the dataset with more examples covering different machining operations and controller styles. I would also compare the base model and fine-tuned model more systematically to quantify the improvements in consistency and accuracy.